# Дообучение PP-OCRv5 rec (eslav) под русские чеки

Этот ноутбук дообучает распознающую модель `eslav_PP-OCRv5_mobile_rec` на синтетических строках чеков и экспортирует её в ONNX для сервиса `Skinemsya_ml`.

**Перед запуском (локально, на своём компьютере):**

1. Сгенерируйте датасет: `python -m training.synthgen --count 3000 --out data/synth`
2. Заархивируйте: `cd data/synth && zip -r ../synth.zip lines rec_gt.txt && cd ../..` (получится `data/synth.zip`)

**В Colab:** меню «Среда выполнения» → «Сменить среду выполнения» → выберите GPU (T4). Затем выполняйте ячейки по порядку (Shift+Enter). Подробная инструкция для первого запуска — в `docs/colab-guide.md`.

Ожидаемое время: установка ~5 мин, обучение на 3000 чеков (~65 тыс. строк) — 30–60 мин на T4.

In [ ]:
# Шаг 1. Проверяем, что нам выдали GPU (должна появиться таблица с Tesla T4).
# Если команда падает — включите GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU.
!nvidia-smi

In [ ]:
# Шаг 2. Устанавливаем PaddlePaddle (GPU) и клонируем PaddleOCR (~5 минут).
# Если pip не находит paddlepaddle-gpu под текущую CUDA, посмотрите актуальную
# команду на https://www.paddlepaddle.org.cn/en/install/quick
%cd /content
!pip install -q "paddlepaddle-gpu" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ || pip install -q paddlepaddle-gpu
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd /content/PaddleOCR
!pip install -q -r requirements.txt
!pip install -q paddle2onnx

import paddle
print("Paddle:", paddle.__version__, "| CUDA доступна:", paddle.device.is_compiled_with_cuda())

In [ ]:
# Шаг 3. Загружаем датасет (synth.zip, подготовленный локально).
# Появится кнопка "Выбрать файлы" — выберите data/synth.zip со своего компьютера.
# Для больших архивов удобнее Google Drive: раскомментируйте блок ниже.
from google.colab import files

uploaded = files.upload()  # выберите synth.zip
zip_name = next(iter(uploaded))

# --- Вариант с Google Drive (положите synth.zip в корень Диска) ---
# from google.colab import drive
# drive.mount('/content/drive')
# zip_name = '/content/drive/MyDrive/synth.zip'

!mkdir -p /content/dataset
!unzip -qo "{zip_name}" -d /content/dataset
!ls /content/dataset && wc -l /content/dataset/rec_gt.txt

In [ ]:
# Шаг 4. Делим датасет на train/val (95/5) и находим словарь символов eslav.
import glob
import pathlib
import random

lines = pathlib.Path("/content/dataset/rec_gt.txt").read_text(encoding="utf-8").splitlines()
random.Random(42).shuffle(lines)
split = int(len(lines) * 0.95)
pathlib.Path("/content/dataset/rec_gt_train.txt").write_text("\n".join(lines[:split]) + "\n", encoding="utf-8")
pathlib.Path("/content/dataset/rec_gt_val.txt").write_text("\n".join(lines[split:]) + "\n", encoding="utf-8")
print(f"train: {split}, val: {len(lines) - split}")

# Словарь символов модели eslav (восточнославянские языки).
dict_candidates = glob.glob("/content/PaddleOCR/ppocr/utils/dict/**/*eslav*", recursive=True)
assert dict_candidates, "Словарь eslav не найден — проверьте версию PaddleOCR"
DICT_PATH = dict_candidates[0]
print("Словарь:", DICT_PATH)

# Проверяем, что все символы датасета есть в словаре (отсутствующие модель не выучит).
dict_chars = set(pathlib.Path(DICT_PATH).read_text(encoding="utf-8").split("\n"))
dataset_chars = set()
for line in lines:
    if "\t" in line:
        dataset_chars.update(line.split("\t", 1)[1])
missing = dataset_chars - dict_chars - {" "}
print("Символы вне словаря:", missing if missing else "нет — всё покрыто")

# Конфиг обучения eslav-модели (если его нет в репо — берём базовый PP-OCRv5 mobile).
cfg_candidates = glob.glob("/content/PaddleOCR/configs/rec/**/*eslav*", recursive=True)
CONFIG_PATH = cfg_candidates[0] if cfg_candidates else "configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml"
print("Конфиг:", CONFIG_PATH)

PRETRAINED_URL = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/eslav_PP-OCRv5_mobile_rec_pretrained.pdparams"
!wget -q {PRETRAINED_URL} -O /content/eslav_pretrained.pdparams && ls -lh /content/eslav_pretrained.pdparams

In [ ]:
# Шаг 5. Обучение (~30-60 минут на T4).
# Дообучаем с претрейна: маленький learning rate, немного эпох.
# Метрика acc в логах — доля полностью верно распознанных строк на валидации.
!cd /content/PaddleOCR && python tools/train.py -c {CONFIG_PATH} \
    -o Global.pretrained_model=/content/eslav_pretrained.pdparams \
       Global.character_dict_path={DICT_PATH} \
       Global.max_text_length=40 \
       Global.epoch_num=10 \
       Global.save_model_dir=/content/output/receipt_rec \
       Global.eval_batch_step="[0, 500]" \
       Global.print_batch_step=50 \
       Optimizer.lr.learning_rate=0.0005 \
       Train.dataset.data_dir=/content/dataset \
       Train.dataset.label_file_list="['/content/dataset/rec_gt_train.txt']" \
       Train.loader.batch_size_per_card=128 \
       Eval.dataset.data_dir=/content/dataset \
       Eval.dataset.label_file_list="['/content/dataset/rec_gt_val.txt']" \
       Eval.loader.batch_size_per_card=128

In [ ]:
# Шаг 6. Экспорт лучшего чекпоинта в inference-формат и конвертация в ONNX.
!cd /content/PaddleOCR && python tools/export_model.py -c {CONFIG_PATH} \
    -o Global.pretrained_model=/content/output/receipt_rec/best_accuracy \
       Global.character_dict_path={DICT_PATH} \
       Global.max_text_length=40 \
       Global.save_inference_dir=/content/inference/receipt_rec

!ls /content/inference/receipt_rec

# Новый PIR-формат Paddle 3.x — inference.json; старый — inference.pdmodel.
import os
model_file = ("inference.json" if os.path.exists("/content/inference/receipt_rec/inference.json")
              else "inference.pdmodel")
!cd /content && paddle2onnx \
    --model_dir /content/inference/receipt_rec \
    --model_filename {model_file} \
    --params_filename inference.pdiparams \
    --save_file /content/receipt_rec.onnx \
    --opset_version 14
!ls -lh /content/receipt_rec.onnx

In [ ]:
# Шаг 7. Скачиваем результат: модель + словарь символов.
import shutil
shutil.copy(DICT_PATH, "/content/receipt_rec_keys.txt")

from google.colab import files
files.download("/content/receipt_rec.onnx")
files.download("/content/receipt_rec_keys.txt")

## Подключение модели к сервису

1. Положите скачанные файлы в репозиторий `Skinemsya_ml`: `data/models/receipt_rec.onnx` и `data/models/receipt_rec_keys.txt`.
2. Проверьте качество локально: `python scripts/evaluate.py --dataset data/synth_eval --rec-model data/models/receipt_rec.onnx --rec-keys data/models/receipt_rec_keys.txt` (сравнит метрики с базовой моделью).
3. Для запуска сервиса с новой моделью задайте переменные окружения:

```bash
ML_REC_MODEL_PATH=data/models/receipt_rec.onnx
ML_REC_KEYS_PATH=data/models/receipt_rec_keys.txt
```

В Docker — раскомментируйте volume и переменные в `docker-compose.yml`.